[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/05_tag_6_8_conv2d_filter_featuremaps_detail.ipynb)

# Tag 6-8 - 05 Conv2d im Detail: Filter, Aktivierungen, Feature Maps

`nn.Conv2d` speichert Filter als trainierbare Gewichte. Dieses Beispiel zeigt Gewichtstensor, Bias, manuell gesetzte Filter, ReLU und Feature Maps.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from sklearn.datasets import load_digits


## Gewichtstensor und Parameteranzahl


In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, padding=1).to(device)
print("Gewichte:", conv.weight.shape)
print("Bias:", conv.bias.shape)
print("Parameter:", sum(p.numel() for p in conv.parameters()))


## Filter setzen und anzeigen


In [ ]:
manual_filters = torch.tensor([
    [[[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]],
    [[[-1, -1, -1], [0, 0, 0], [1, 1, 1]]],
    [[[1/9, 1/9, 1/9], [1/9, 1/9, 1/9], [1/9, 1/9, 1/9]]],
    [[[0, -1, 0], [-1, 5, -1], [0, -1, 0]]],
], dtype=torch.float32, device=device)
with torch.no_grad():
    conv.weight[:4] = manual_filters
    conv.bias.zero_()

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(conv.weight[i, 0].detach().cpu(), cmap="coolwarm")
    ax.set_title(f"Filter {i}")
    ax.axis("off")


## Feature Maps vor und nach ReLU


In [ ]:
digits = load_digits()
idx = 7
X = torch.tensor(digits.images[idx:idx + 1], dtype=torch.float32).unsqueeze(1).to(device) / 16.0
label = digits.target[idx]
with torch.no_grad():
    pre_relu = conv(X)
    post_relu = torch.relu(pre_relu)

fig = plt.figure(figsize=(14, 4))
grid = fig.add_gridspec(2, 7, width_ratios=[1.2] + [1] * 6)

# Das Originalbild bleibt als Referenz neben allen Feature Maps sichtbar.
original_ax = fig.add_subplot(grid[:, 0])
original_ax.imshow(X[0, 0].detach().cpu(), cmap="gray")
original_ax.set_title(f"Original\nLabel: {label}")
original_ax.axis("off")

for i in range(6):
    before_ax = fig.add_subplot(grid[0, i + 1])
    before_ax.imshow(pre_relu[0, i].detach().cpu(), cmap="viridis")
    before_ax.set_title(f"vor ReLU {i}")
    before_ax.axis("off")

    after_ax = fig.add_subplot(grid[1, i + 1])
    after_ax.imshow(post_relu[0, i].detach().cpu(), cmap="viridis")
    after_ax.set_title(f"nach ReLU {i}")
    after_ax.axis("off")

plt.tight_layout()
